# Fine Tune QLoRa Phi-3-mini with Vietnamese news dataset to summary

## Setting Enviroments

In [1]:
!nvidia-smi

Fri Dec 12 23:48:22 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             41W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# hide output
%%capture
# Cài đặt thư viện Hugging Face CLI
!pip install huggingface_hub


In [3]:
from huggingface_hub import notebook_login
notebook_login()

In [4]:
# hide output
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install evaluate rouge-score bert-score
import os
os.environ["WANDB_DISABLED"] = "true"

In [5]:
from unsloth import FastLanguageModel
import torch

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from datasets import load_dataset

from peft import LoraConfig, get_peft_model, TaskType, PeftModel, PeftConfig
from rouge_score import rouge_scorer  # SỬA: Import cho ROUGE
import bert_score

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


🦥 Unsloth Zoo will now patch everything to make training faster!


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using

## Utils

In [6]:
# ROUGE function
def evaluate_rouge(reference, prediction):
  scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
  scores = scorer.score(reference, prediction)
  return {
      'rouge1' : scores['rouge1'].fmeasure,
      'rouge2' : scores['rouge2'].fmeasure,
      'rougeL' : scores['rougeL'].fmeasure
  }

In [7]:
# BERT score function
def evaluate_bertscore(reference, prediction, lang='vi'):
  P, R, F1 = score(prediction, reference, lang=lang, verbose=False)
  return {
    'bert_precision': P.mean().item(),
    'bert_recall': R.mean().item(),
    'bert_f1': F1.mean().item()
  }


In [8]:
# function compute_metrics for Trainer
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = torch.argmax(torch.from_numpy(logits), dim=-1).tolist()  # Decode predictions nếu cần
  references = labels.tolist()

  # Giả sử bạn decode predictions và references thành text (cần tokenizer)
  # Ở đây tôi giả định predictions và references đã là list string; thực tế cần decode
  decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
  decoded_refs = tokenizer.batch_decode(references, skip_special_tokens=True)

  # Tính ROUGE (trung bình)
  rouge_scores = {'rouge1': 0, 'rouge2': 0, 'rougeL': 0}
  for ref, pred in zip(decoded_refs, decoded_preds):
      rouge = evaluate_rouge(ref, pred)
      for k in rouge_scores:
          rouge_scores[k] += rouge[k]
  for k in rouge_scores:
      rouge_scores[k] /= len(decoded_preds)

  # Tính BERTScore
  # bert_scores = evaluate_bertscore(decoded_refs, decoded_preds)
  return {**rouge_scores}

  # return {**rouge_scores, **bert_scores}

In [9]:
import numpy as np
from evaluate import load

# tải bộ đánh giá ROUGE (batch compute, nhanh hơn)
rouge_metric = load("rouge")

def compute_metrics(eval_pred):
    """
    eval_pred: tuple (logits, labels) từ Trainer
    - logits: numpy array, shape (batch, seq_len, vocab_size) cho causal LM
    - labels: numpy array, shape (batch, seq_len) có -100 cho padding/masked tokens
    """
    logits, labels = eval_pred

    # 1) Lấy prediction token ids: argmax theo vocab
    # Nếu logits là None hoặc đã là logits của thằng generate thì cần xử lý khác.
    if logits is None:
        raise ValueError("logits is None in eval_pred")

    # logits shape: (batch, seq_len, vocab_size)
    preds_ids = np.argmax(logits, axis=-1)

    # 2) Thay -100 bằng pad_token_id trước khi decode
    pad_id = getattr(tokenizer, "pad_token_id", None)
    if pad_id is None:
        # nếu tokenizer không có pad_token, fallback dùng eos_token
        pad_id = tokenizer.eos_token_id

    labels_for_decode = np.where(labels != -100, labels, pad_id)

    # 3) Decode lists of token ids -> strings
    decoded_preds = tokenizer.batch_decode(preds_ids, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels_for_decode, skip_special_tokens=True)

    # 4) Clean & strip
    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    # 5) Tính ROUGE bằng evaluate (hỗ trợ batch và stemmer)
    rouge_result = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    # rouge_result keys: 'rouge1','rouge2','rougeL' (floats)
    # đảm bảo convert sang float Python
    metrics = {
        "rouge1": float(rouge_result.get("rouge1", 0.0)),
        "rouge2": float(rouge_result.get("rouge2", 0.0)),
        "rougeL": float(rouge_result.get("rougeL", 0.0)),
    }

    return metrics


In [10]:
model = None
tokenizer = None

def summarize(text, prompt, model=model, tokenizer=tokenizer,
              max_new_tokens=1000, temperature=0.2):

  prompt = f"""<|user|>
  {prompt}

  Văn bản: {text}
  <|end|>
  <|assistant|>
  """

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

  generated_ids = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      temperature=temperature,
      do_sample=False,  # để chính xác, không ngẫu nhiên
      pad_token_id=tokenizer.eos_token_id,
  )

  # 4. Decode output
  decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=False)

  assistant_tag = "<|assistant|>"
  if assistant_tag in decoded:
      summary = decoded.split(assistant_tag, 1)[1]
  else:
      summary = decoded

  summary = summary.replace("<|end|>", "").strip()

  return summary


## Load model

In [11]:
model_name = "unsloth/Phi-3-mini-4k-instruct"
output_dir = './Phi3-4B-QLoRa-summary'
max_seq_length = 2048 # unsloth auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

In [12]:
def get_model_tokenizer(model_name, max_seq_length = 2048, dtype=None, load_in_4bit=True):
  # 4bit pre quantized models

  model, tokenizer = FastLanguageModel.from_pretrained(
      model_name = model_name,
      max_seq_length = max_seq_length,
      dtype = dtype,
      load_in_4bit = load_in_4bit,
  )

  # model = PeftModel.from_pretrained( # load fine continue checkpoint
  #   model,
  #   "/content/drive/MyDrive/lora_checkpt",
  # )

  return model, tokenizer

In [13]:
bnb_4bit_compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 8 else torch.float16

model, tokenizer = get_model_tokenizer(model_name, max_seq_length, bnb_4bit_compute_dtype, load_in_4bit)

==((====))==  Unsloth 2025.12.5: Fast Mistral patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
model

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((3072,), eps=1e-05)
        (post_attention_laye

## Load data

In [15]:
dataset = load_dataset("Hoang42/summary_article")

README.md:   0%|          | 0.00/197 [00:00<?, ?B/s]

data/train.jsonl:   0%|          | 0.00/716M [00:00<?, ?B/s]

data/validation.jsonl:   0%|          | 0.00/89.2M [00:00<?, ?B/s]

data/test.jsonl:   0%|          | 0.00/89.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/220520 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/27565 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/27565 [00:00<?, ? examples/s]

In [16]:
val = dataset["validation"]
old_train = dataset["train"]
# new_train = old_train.shuffle(seed=42).select(range(10000))
# new_eval = val.shuffle(seed=42).select(range(500))
new_train = old_train
new_eval = val

In [17]:
new_train[0]

{'Intruction': 'Tóm tắt văn bản sau:',
 'input': 'Tiêu đề: Đà Nẵng đảm bảo 100% thí sinh được dự thi Tốt nghiệp THPT\n\nNội dung: Theo thống kê, cả nước có 26.014 thí sinh của 27 tỉnh, thành trên cả nước chưa thể dự thi Tốt nghiệp THPT đợt 1. Riêng với Đà Nẵng, vùng tâm dịch với hơn 10.000 thí sinh. Nhằm đảm bảo cho công tác phòng chống dịch trong kì thi THPT sắp tới, tất cả các thí sinh đều được lấy mẫu xét nghiệm trước khi ngày thi chính thức bắt đầu. Riêng với các thí sinh thuộc diện F1, F2 và đang ở khu vực cách ly y tế sẽ được Trung tâm Kiểm soát bệnh tật thành phố đến lấy mẫu tại nhà hoặc nơi đang cách ly. Ông Trần Nguyễn Minh Thành – Phó Giám đốc Sở Giáo dục Đà Nẵng cho biết, 24 hội đồng thi với hơn 10.000 thí sinh được hẹn lịch lấy mẫu trong sáng và chiều ngày 31.8. Ngày mai (1.9), toàn thể cán bộ nhân viên, giáo viên làm nhiệm vụ coi thi sẽ được lấy mẫu xét nghiệm. Đặc biệt, nhằm đảm bảo cho 100% các thí sinh tại Đà Nẵng có cơ hội được tham dự kì thi, ông Thành cho biết: “Tron

In [18]:
def format_instruction(example):
  # Sử dụng 'Intruction' từ dataset làm System Prompt (hoặc fallback nếu không có)
  system_prompt = example.get('Intruction', "Bạn là trợ lý AI chuyên tóm tắt văn bản tiếng Việt. Tóm tắt nội dung sau, giữ ý chính.")

  # Tạo chuỗi huấn luyện hoàn chỉnh theo chuẩn Instruct
  # Keys được sử dụng là 'input' (nội dung) và 'output' (tóm tắt)
  full_text = f"<|system|>\n{system_prompt}<|end|>\n<|user|>\n{example['input']}<|end|>\n<|assistant|>\n{example['output']}<|end|>"

  return {"text": full_text}

In [19]:
# Apply format
new_train = new_train.map(format_instruction, num_proc=2, remove_columns=['Intruction', 'input', 'output'])
new_eval = new_eval.map(format_instruction, num_proc=2, remove_columns=['Intruction', 'input', 'output'])

Map (num_proc=2):   0%|          | 0/220520 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/27565 [00:00<?, ? examples/s]

In [20]:
new_train[0]

{'text': '<|system|>\nTóm tắt văn bản sau:<|end|>\n<|user|>\nTiêu đề: Đà Nẵng đảm bảo 100% thí sinh được dự thi Tốt nghiệp THPT\n\nNội dung: Theo thống kê, cả nước có 26.014 thí sinh của 27 tỉnh, thành trên cả nước chưa thể dự thi Tốt nghiệp THPT đợt 1. Riêng với Đà Nẵng, vùng tâm dịch với hơn 10.000 thí sinh. Nhằm đảm bảo cho công tác phòng chống dịch trong kì thi THPT sắp tới, tất cả các thí sinh đều được lấy mẫu xét nghiệm trước khi ngày thi chính thức bắt đầu. Riêng với các thí sinh thuộc diện F1, F2 và đang ở khu vực cách ly y tế sẽ được Trung tâm Kiểm soát bệnh tật thành phố đến lấy mẫu tại nhà hoặc nơi đang cách ly. Ông Trần Nguyễn Minh Thành – Phó Giám đốc Sở Giáo dục Đà Nẵng cho biết, 24 hội đồng thi với hơn 10.000 thí sinh được hẹn lịch lấy mẫu trong sáng và chiều ngày 31.8. Ngày mai (1.9), toàn thể cán bộ nhân viên, giáo viên làm nhiệm vụ coi thi sẽ được lấy mẫu xét nghiệm. Đặc biệt, nhằm đảm bảo cho 100% các thí sinh tại Đà Nẵng có cơ hội được tham dự kì thi, ông Thành cho 

## Set up QLoRa

In [21]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0,
    bias="none",
    use_rslora=True,
    use_gradient_checkpointing="unsloth"  # nếu muốn checkpointing
)


Unsloth 2025.12.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [22]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj

In [23]:
text = """
Tiêu đề: Đảm bảo phúc lợi cho người lao động trong dịch bệnh COVID-19

Nội dung: Buổi toạ đàm là dịp để các Công đoàn cơ sở tại doanh nghiệp được chia sẻ, trao đổi những kinh nghiệm thực tế, khó khăn và nỗ lực, từ đó rút ra bài học trong quá trình hoạt động ở cơ sở. Tại buổi tọa đàm, các đại biểu đã trao đổi nhiều kinh nghiệm về vận động doanh nghiệp duy trì, đảm bảo những điều khoản phúc lợi trong thỏa ước lao động tập thể; giải quyết chế độ, chính sách theo luật quy định của người lao động khi doanh nghiệp thu hẹp sản xuất, cắt giảm lao động. Đồng thời, đối thoại với chủ sử dụng lao động nhằm duy trì việc làm, thu nhập của người lao động trong bối cảnh dịch bệnh diễn biến phức tạp thời gian qua. Bà Đoàn Thị Kim Loan – Công đoàn cơ sở Công ty Changshin Việt Nam (xã Thạnh Phú, huyện Vĩnh Cửu, Đồng Nai) cho biết: Trong thời gian dịch bệnh, Công đoàn công ty đã có nhiều sáng kiến cùng Ban giám đốc công ty đảm bảo được việc phòng chống dịch bệnh, đồng thời đảm bảo được quyền lợi đầy đủ cho người lao động. Do đó, người lao động của công ty không bị nghỉ việc trong dịch COVID-19. Theo LĐLĐ tỉnh, từ đầu năm đến nay do ảnh hưởng của dịch bệnh COVID-19, toàn tỉnh có 129 doanh nghiệp và trên 100.000 người lao động gặp khó khăn, trong đó có trên 82.000 người lao động bị giảm giờ làm trong tuần hoặc nghỉ không hưởng lương, chấm dứt hợp đồng lao động. Trong đó, các nhóm ngành nghề ảnh hưởng nhiều nhất như: dệt may, giày da, sản xuất đồ gỗ, điện tử… Với những khó khăn nói trên, các cấp công đoàn đã có nhiều nỗ lực nhằm tuyên truyền, hướng dẫn, đối thoại, chia sẻ thông tin với doanh nghiệp để ổn định sản xuất, đảm bảo việc làm và các chế độ chính sách cho người lao động.
"""

prompt = "Hãy tóm tắt bài báo này thành 3 câu sau cho tổng quát hết được nội dung"

# summary = summarize(text, prompt, model, tokenizer, 1500)
# print(summary)

## Train

In [24]:
 # @title GPU T4 colab free

# HUB_MODEL_ID = "Hoang42/Phi3-4B-QLoRa-summary-adapter"
# trainer = SFTTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = new_train,
#     eval_dataset = new_eval,
#     dataset_text_field = "text",
#     max_seq_length = max_seq_length,
#     dataset_num_proc = 2,
#     packing = True, # Can make training 5x faster for short sequences.
#     # padding_free_opt = False,
#     compute_metrics = compute_metrics,
#     args = TrainingArguments(
#       per_device_train_batch_size=8,
#       per_device_eval_batch_size=1,
#       gradient_accumulation_steps = 4,
#       warmup_steps = 5,
#       # max_steps = 10,

#       max_steps = -1,
#       num_train_epochs=1, # train_epochs

#       learning_rate = 1e-4,
#       fp16 = not is_bfloat16_supported(),
#       bf16 = is_bfloat16_supported(),
#       logging_steps = 1,
#       optim = "adamw_torch_fused",
#       weight_decay = 0.01,
#       lr_scheduler_type = "cosine",
#       seed = 42,

#       save_strategy="steps",
#       save_steps=50,
#       save_total_limit=2,
#       save_safetensors=True,

#       output_dir = "outputs",
#       report_to="none",

#       push_to_hub=True,
#       hub_model_id=HUB_MODEL_ID,
#       hub_strategy='checkpoint',
#     ),
# )

In [25]:
# @title GPU A100 80GB VRAM, 167GB RAM

HUB_MODEL_ID = "Hoang42/Phi3-4B-QLoRa-summary-adapter"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = new_train,
    eval_dataset = new_eval,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True,
    compute_metrics = compute_metrics,
    args = TrainingArguments(
      per_device_train_batch_size = 128,
      per_device_eval_batch_size = 1,
      gradient_accumulation_steps = 1,
      warmup_steps = 100,

      max_steps = -1,
      num_train_epochs=1,

      learning_rate = 1e-4,
      fp16 = False,
      bf16 = True,                         # A100
      logging_steps = 10,
      # optim = "paged_adamw_8bit",
      optim = 'adamw_torch_fused',
      weight_decay = 0.01,
      lr_scheduler_type = "cosine",
      seed = 42,

      save_strategy="steps",
      save_steps=50,
      save_total_limit=2,
      save_safetensors=True,

      output_dir = "outputs",
      report_to="none",

      push_to_hub=True,
      hub_model_id=HUB_MODEL_ID,
      hub_strategy='checkpoint', # Đẩy lên Hub sau mỗi lần save_steps
  ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/220520 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/27565 [00:00<?, ? examples/s]

In [26]:
#Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.557 GB.
3.572 GB of memory reserved.


In [27]:
#@title Train start
# trainer_stats = trainer.train()


In [28]:
# @title Reusume checkPoint

from huggingface_hub import snapshot_download

local_repo = snapshot_download(
  repo_id="Hoang42/Phi3-4B-QLoRa-summary-adapter",
  revision="main"
)

checkpoint_path = local_repo + "/last-checkpoint"

trainer.train(resume_from_checkpoint=checkpoint_path)
# trainer.train(resume_from_checkpoint=True)



Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/120M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/407 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

...

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 220,520 | Num Epochs = 1 | Total steps = 1,723
O^O/ \_/ \    Batch size per device = 128 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (128 x 1 x 1) = 128
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1210,0.981700
1220,0.972100
1230,0.966600
1240,0.973600
1250,0.979600
1260,0.983600
1270,0.969900
1280,0.964500
1290,0.972200
1300,0.977900


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using

TrainOutput(global_step=1723, training_loss=0.2940522877200011, metrics={'train_runtime': 9828.0987, 'train_samples_per_second': 22.438, 'train_steps_per_second': 0.175, 'total_flos': 5.084118344584397e+18, 'train_loss': 0.2940522877200011, 'epoch': 1.0})

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

metrics = trainer.evaluate()
print(metrics)

In [30]:
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

trainer.push_to_hub(commit_message="End of fine-tuning v1.0")

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...outputs/training_args.bin: 100%|##########| 6.22kB / 6.22kB            

  ...t/outputs/tokenizer.model: 100%|##########|  500kB /  500kB            

  ...adapter_model.safetensors:   1%|1         |  613kB / 59.8MB            

CommitInfo(commit_url='https://huggingface.co/Hoang42/Phi3-4B-QLoRa-summary-adapter/commit/4b565a76586337aac38b2b05a1377ff61543cec0', commit_message='End of fine-tuning v1.0', commit_description='', oid='4b565a76586337aac38b2b05a1377ff61543cec0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Hoang42/Phi3-4B-QLoRa-summary-adapter', endpoint='https://huggingface.co', repo_type='model', repo_id='Hoang42/Phi3-4B-QLoRa-summary-adapter'), pr_revision=None, pr_num=None)

## Inference

In [35]:
prompt = "Hãy tóm tắt bài báo này thành 3 câu sau cho tổng quát hết được nội dung"

summary = summarize(text, prompt, model, tokenizer, 2000)
print(summary)

Công đoàn cơ sở Công ty Changshin Việt Nam (xã Thạnh Phú, huyện Vĩnh Cửu, Đồng Nai) đã tổ chức buổi tọa đàm “Tạo điều kiện cho người lao động tham gia thực hiện các chế độ chính sách theo quy định của pháp luật và các quy định của doanh nghiệp” vào ngày 14.10.


## Evaluate

In [32]:
import json
# Đảm bảo new_eval (10 mẫu đã được format) và summarize đã sẵn sàng

predictions = []
references = []

# Tải lại dataset gốc (chỉ cần output) vì new_eval đã bị map qua format_instruction
dataset_val_raw = load_dataset("Hoang42/summary_article")["validation"]
new_eval_raw = dataset_val_raw.shuffle(seed=42).select(range(10))

# prompt_text = new_eval_raw[0]['Intruction'] # Lấy instruction từ mẫu đầu tiên
prompt_text = "Hãy tóm tắt bài báo này thành 3 câu sau cho tổng quát hết được nội dung"

print("Generating predictions...")
for example in new_eval_raw:
    # 1. Lấy prediction bằng hàm summarize đã define (dùng mô hình QLoRa đã train)
    article = example['input']
    generated_summary = summarize(prompt_text, prompt, model, tokenizer, 2000)
    predictions.append(generated_summary.replace("\n", " ").strip())

    # 2. Lấy reference (gold summary)
    references.append(example['output'].replace("\n", " ").strip())

# 3. Tạo dictionary và lưu ra file JSON
bertscore_data = {
    "predictions": predictions,
    "references": references,
}

# LƯU VÀO THƯ MỤC CHUNG MÀ CẢ HAI KERNEL CÓ THỂ TRUY CẬP
output_file_path = "bertscore_input_data.json"
with open(output_file_path, "w", encoding="utf-8") as f:
    json.dump(bertscore_data, f, ensure_ascii=False, indent=4)

print(f"Đã lưu {len(predictions)} cặp dữ liệu vào {output_file_path}")

# HÃY NHỚ KHỞI ĐỘNG LẠI KERNEL/RUNTIME (Ngắt kết nối và xóa phiên runtime)
# TRƯỚC KHI CHUYỂN SANG PHẦN 2!

Generating predictions...
Đã lưu 10 cặp dữ liệu vào bertscore_input_data.json


In [33]:
from google.colab import files

# Tên file bạn muốn tải về
output_file_path = "bertscore_input_data.json"

# Lệnh tải file
files.download(output_file_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
# # CODE NÀY CHẠY TRÊN KERNEL MỚI (TPU/CPU/47GB RAM)

# # 1. Cài đặt thư viện
# # %%capture
# !pip install bert-score datasets

# import os
# import json
# from bert_score import score
# import torch

# # 2. Định nghĩa hàm tính BERTScore (buộc dùng CPU)
# def calculate_bertscore_cpu(references, predictions, lang='vi', batch_size=4):
#     """Tính BERTScore, buộc chạy trên CPU."""
#     print(f"Đang tải và tính BERTScore trên CPU...")

#     # Cấu hình thiết bị là CPU
#     device = 'cpu'

#     # Tính toán
#     # Sử dụng mBERT là lựa chọn phổ biến cho tiếng Việt
#     P, R, F1 = score(
#         predictions,
#         references,
#         lang=lang,
#         verbose=True,
#         device=device,
#         batch_size=batch_size,
#     )

#     # Tính trung bình các chỉ số
#     avg_metrics = {
#       'bert_precision': P.mean().item(),
#       'bert_recall': R.mean().item(),
#       'bert_f1': F1.mean().item()
#     }

#     return avg_metrics

# # 3. Tải Dữ liệu đã lưu từ Phần 1
# input_file_path = "bertscore_input_data.json"

# try:
#     with open(input_file_path, "r", encoding="utf-8") as f:
#         bertscore_data = json.load(f)

#     predictions = bertscore_data["predictions"]
#     references = bertscore_data["references"]

#     print(f"Đã tải thành công {len(predictions)} cặp dữ liệu.")

#     # 4. Thực hiện Tính toán
#     if len(predictions) > 0:
#         avg_scores = calculate_bertscore_cpu(references, predictions, lang='vi')

#         print("\n--- KẾT QUẢ BERT SCORE TRUNG BÌNH ---")
#         print(f"Precision (P): {avg_scores['bert_precision']:.4f}")
#         print(f"Recall (R): {avg_scores['bert_recall']:.4f}")
#         print(f"F1 Score (F1): {avg_scores['bert_f1']:.4f}")
#         print("---------------------------------------")

# except FileNotFoundError:
#     print(f"Lỗi: Không tìm thấy file {input_file_path}. Hãy đảm bảo file đã được lưu từ Phần 1.")